In [ ]:
import time
GLOBAL_START_TIME = time.time()

In [ ]:
!nvidia-smi

In [ ]:
!uv pip uninstall 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import os
import sys
import subprocess
import warnings
warnings.simplefilter('ignore')

In [ ]:
!uv pip install --no-index --find-links /kaggle/input/notebooks/chankhavu/fork-of-vllm-0-18-0-py312-d168-flashinfer066/wheels vllm flashinfer-jit-cache flashinfer-python

In [ ]:
# def set_env_vllm_mini(input_archive):
#     install_libraries = [
#         'pip', 'install', '--no-index', '--find-links', f'{input_archive}/wheels',
#         'vllm', 'flashinfer-jit-cache', 'flashinfer-python', # 'flashinfer-cubin',
#     ]
#     subprocess.run(install_libraries, check=True)
# 
# 
# input_whl='/kaggle/input/notebooks/chankhavu/fork-of-vllm-0-18-0-py312-d168-flashinfer066'
# set_env_vllm_mini(input_archive=input_whl)
print("WALL TIME ELAPSED:", time.time() - GLOBAL_START_TIME)

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['VLLM_USE_FLASHINFER_MOE_FP8'] = '1'

In [ ]:
import asyncio
import gc
import re
import json
import math
import time
import threading
from typing import Optional
from uuid import uuid4
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import polars as pl

from openai import OpenAI

import kaggle_evaluation.aimo_3_inference_server

In [ ]:
!mkdir scripts
!touch scripts/__init__.py
!cp /kaggle/input/notebooks/chankhavu/ipython-sandbox/sandbox_client.py ./scripts/sandbox_client.py
!cp /kaggle/input/notebooks/chankhavu/ipython-sandbox/sandbox_server.py ./scripts/sandbox_server.py

from scripts.sandbox_client import LocalSandbox

In [ ]:
TOOLS = [{
    "type": "function",
    "function": {
        "name": "stateful_python_code_exec",
        "description": (
            "Call this function to execute Python code in a stateful Jupyter notebook environment. "
            "Python will respond with the output of the execution or time out after 120.0 seconds. "
            "The environment has math, numpy, sympy, itertools, collections, and mpmath pre-imported."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": "Code to execute",
                }
            },
            "required": ["code"],
        },
    },
}]

In [ ]:
class CFG:

    system_prompt = (
        'You are a helpful assistant and a mathematical problem solver. '
        'Solve the given problem step by step using rigorous reasoning. '
    )

    user_template = (
        '{problem}\n\nThe answer is a non-negative integer between 0 and 99999, '
        'inclusive. Put your final answer in \\boxed{{}}.'
    )

    served_model_name = 'chankhavu/c2-legaldistill-30b-a3b-fp8'
    model_path = '/kaggle/input/models/chankhavu/bogoconic1-c2-softcpy-v3-fp8/transformers/v1/1'

    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17600
    server_timeout = 180

    vllm_port = 8000
    session_timeout = 960
    sandbox_host = '127.0.0.1'
    sandbox_port = 6000
    python_timeout = 30
    max_output_characters = 1000

    context_tokens = 131072
    max_completion_tokens = 131072
    buffer_tokens = 512
    top_logprobs = 5
    max_num_seqs = 64
    early_stop = 4
    attempts = 16
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.9
    temperature = 1.0
    top_p = 0.95

    mamba_ssm_cache_dtype = 'float16'
    mamba_cache_mode = 'all'
    mamba_block_size = 512
    max_num_batched_tokens = 8192

    enable_thinking = True

In [ ]:
SANDBOX_PREIMPORT = (
    "import math\n"
    "import numpy\n"
    "import sympy\n"
    "import itertools\n"
    "import collections\n"
    "import mpmath\n"
    "mpmath.mp.dps = 64\n"
)


In [ ]:
def _scan_for_answer(text: str) -> int | None:

    pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
    matches = re.findall(pattern, text)

    if matches:
        try:
            clean_value = matches[-1].replace(',', '')
            value = int(clean_value)

            if 0 <= value <= 99999:
                return value

        except ValueError:
            pass

    # Fallback: \boxed{...expr = DIGITS}
    boxed_contents = re.findall(r'\\boxed\s*\{([^}]+)\}', text)
    if boxed_contents:
        nums = re.findall(r'(\d[\d,]*)', boxed_contents[-1])
        if nums:
            try:
                value = int(nums[-1].replace(',', ''))
                if 0 <= value <= 99999:
                    return value
            except ValueError:
                pass

    pattern = r'final\s+answer\s+is\s*([0-9,]+)'
    matches = re.findall(pattern, text, re.IGNORECASE)

    if matches:
        try:
            clean_value = matches[-1].replace(',', '')
            value = int(clean_value)

            if 0 <= value <= 99999:
                return value

        except ValueError:
            pass

    return None


def _strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from text."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

In [ ]:
sandbox_server_script = "scripts/sandbox_server.py"

print('Starting sandbox server...')
sandbox_log_file = open('sandbox_server.log', 'w')
sandbox_process = subprocess.Popen(
    [sys.executable, sandbox_server_script],
    stdout=sandbox_log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# Wait for sandbox to be ready
_sandbox_check = LocalSandbox(host=CFG.sandbox_host, port=CFG.sandbox_port)

for _attempt in range(30):
    rc = sandbox_process.poll()
    if rc is not None:
        sandbox_log_file.flush()
        with open('sandbox_server.log', 'r') as f:
            logs = f.read()
        raise RuntimeError(f'Sandbox server died with code {rc}. Logs:\n{logs}\n')
    if _sandbox_check._check_ready(timeout=2):
        break
    time.sleep(1)
else:
    raise RuntimeError('Sandbox server failed to start (timeout).\n')

print('Sandbox server is ready.\n')

In [ ]:
def _preload_model_weights():

    print(f'Loading model weights from {CFG.model_path} into OS Page Cache...')
    start_time = time.time()

    files_to_load = []
    total_size = 0

    for root, _, files in os.walk(CFG.model_path):
        for file_name in files:
            file_path = os.path.join(root, file_name)

            if os.path.isfile(file_path):
                files_to_load.append(file_path)
                total_size += os.path.getsize(file_path)

    def _read_file(path: str) -> None:

        with open(path, 'rb') as file_object:
            while file_object.read(1024 * 1024 * 1024):
                pass

    with ThreadPoolExecutor(max_workers=CFG.workers) as executor:
        list(executor.map(_read_file, files_to_load))

    elapsed = time.time() - start_time
    print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

_preload_model_weights()
print("WALL TIME ELAPSED:", time.time() - GLOBAL_START_TIME)

In [ ]:
vllm_cmd = [
    # sys.executable,
    # '-m',
    # 'vllm.entrypoints.openai.api_server',
    "vllm", "serve", CFG.model_path,
    # '--seed',
    # str(CFG.seed),
    # '--model',
    # CFG.model_path,
    '--served-model-name', CFG.served_model_name,
    '--tensor-parallel-size', '1',
    '--max-num-seqs', str(CFG.max_num_seqs),
    '--gpu-memory-utilization', str(CFG.gpu_memory_utilization),
    '--host', '0.0.0.0',
    '--port', str(CFG.vllm_port),
    '--dtype', CFG.dtype,
    '--max-model-len', str(CFG.context_tokens),
    '--max-num-batched-tokens', str(CFG.max_num_batched_tokens),
    '--kv-cache-dtype', 'fp8',
    '--trust-remote-code',
    '--mamba-ssm-cache-dtype', CFG.mamba_ssm_cache_dtype,
    '--mamba-cache-mode', CFG.mamba_cache_mode,
    '--mamba-block-size', str(CFG.mamba_block_size),
    '--enable-chunked-prefill',
    '--enable-prefix-caching',
    '--async-scheduling',
    '--enable-auto-tool-choice',
    '--tool-call-parser', 'qwen3_coder',
]

vllm_log_file = open('vllm_server.log', 'w')
vllm_process = subprocess.Popen(
    vllm_cmd,
    stdout=vllm_log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

openai_client = OpenAI(
    base_url=f'http://0.0.0.0:{CFG.vllm_port}/v1',
    api_key='sk-local',
    timeout=CFG.session_timeout
)

print('Waiting for vLLM server...')
start_time = time.time()

for _ in range(CFG.server_timeout):
    return_code = vllm_process.poll()

    if return_code is not None:
        vllm_log_file.flush()

        with open('vllm_server.log', 'r') as f:
            logs = f.read()

        raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')

    try:
        openai_client.models.list()
        elapsed = time.time() - start_time
        print(f'Server is ready (took {elapsed:.2f} seconds).\n')

        break

    except Exception:
        time.sleep(1)

else:
    raise RuntimeError('Server failed to start (timeout).\n')
print("WALL TIME ELAPSED:", time.time() - GLOBAL_START_TIME)

In [ ]:
# ----------------------------------------------------------------------------
# vLLM watchdog: a background subprocess that re-launches vLLM whenever there
# is no live (non-zombie) vLLM process found in /proc.
#
# Design:
#   - Polls every WATCHDOG_CHECK_INTERVAL seconds.
#   - Each tick scans /proc/*/cmdline for a process containing
#     'vllm.entrypoints', skipping zombie/dead states.
#   - If at least one is found: log "healthy: pids=..." and sleep.
#   - If none: kill any leftover zombies, Popen vLLM again, log RESTART #N.
#   - Reaps its own zombie children at the start of every tick.
#
# This deliberately does not use a PID file. The previous version tracked
# the latest PID in vllm_pid.txt, which combined badly with SIGKILL: the
# child would become a zombie still in /proc, os.kill(pid, 0) would say
# "alive", and the watchdog would never restart. Process scan + state-char
# check is the robust approach.
# ----------------------------------------------------------------------------

WATCHDOG_CHECK_INTERVAL = 30  # seconds — once a minute, as suggested

VLLM_WATCHDOG_CONFIG = 'vllm_watchdog_config.json'
VLLM_WATCHDOG_SCRIPT = 'scripts/vllm_watchdog.py'

with open(VLLM_WATCHDOG_CONFIG, 'w') as _f:
    json.dump({
        'cmd': vllm_cmd,
        'log_path': 'vllm_server.log',
        'health_url': f'http://127.0.0.1:{CFG.vllm_port}/v1/models',
        'check_interval': WATCHDOG_CHECK_INTERVAL,
    }, _f)

# Drop the watchdog script next to sandbox_server.py
_WATCHDOG_SOURCE = '''#!/usr/bin/env python3
"""vLLM watchdog: every check_interval, ensures a healthy vLLM is running.

Per tick:
  1. Reap any of our own zombie children (waitpid -1, WNOHANG).
  2. Scan /proc for non-zombie processes whose cmdline mentions
     'vllm.entrypoints'.
  3. If at least one is found, optionally probe the HTTP health URL just
     for log signal, then sleep.
  4. If none is found, kill any leftover zombie vllm processes, then Popen
     vLLM fresh and log RESTART #N.

The HTTP probe is informational only. Restart is triggered ONLY by "no
live vllm process found", because vLLM startup itself takes ~100s during
which HTTP fails — restarting on HTTP failure would loop forever.
"""
import json
import os
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request


def find_vllm_processes(include_zombies=False):
    """Return PIDs of vLLM processes (APIServer + EngineCore + workers).

    vLLM calls setproctitle("VLLM::APIServer_0") early during startup, which
    overwrites /proc/PID/cmdline. After that, the literal "vllm.entrypoints"
    string is no longer present in cmdline (and it is also wiped to b"" once
    the process becomes a zombie). The 15-char comm name from /proc/PID/stat
    is the reliable signal: it shows "VLLM::APIServer" / "VLLM::EngineCor..."
    for both live and zombie processes. We additionally check cmdline as a
    fallback for the brief window before setproctitle runs.
    """
    pids = []
    try:
        entries = os.listdir("/proc")
    except OSError:
        return pids
    for entry in entries:
        if not entry.isdigit():
            continue
        pid = int(entry)
        try:
            with open("/proc/{}/stat".format(pid)) as f:
                stat = f.read()
            lparen = stat.index("(")
            rparen = stat.rindex(")")
            comm = stat[lparen + 1:rparen]
            state_char = stat[rparen + 2:rparen + 3]

            with open("/proc/{}/cmdline".format(pid), "rb") as f:
                cmdline = f.read().decode("utf-8", errors="ignore")
            argv = [a for a in cmdline.split("\\x00") if a]
            argv0 = os.path.basename(argv[0]) if argv else ""

            cmdline_is_vllm = (
                argv0 == "vllm"
                or any("vllm.entrypoints" in a for a in argv)
            )
            is_vllm = comm.startswith("VLLM::") or cmdline_is_vllm
            if not is_vllm:
                continue
            if not include_zombies and state_char in ("Z", "X"):
                continue
            pids.append(pid)
        except (FileNotFoundError, OSError, ValueError):
            continue
    return pids


def main():
    if len(sys.argv) < 2:
        print("[WATCHDOG] usage: vllm_watchdog.py <config.json>", flush=True)
        sys.exit(2)

    with open(sys.argv[1]) as f:
        config = json.load(f)

    cmd = config["cmd"]
    log_path = config["log_path"]
    health_url = config.get("health_url")
    check_interval = float(config.get("check_interval", 60))

    started_at = time.time()
    state = {"restart_count": 0}

    def elapsed():
        return time.time() - started_at

    def log(msg):
        print("[WATCHDOG t={:.0f}s] {}".format(elapsed(), msg), flush=True)

    def reap_zombies():
        try:
            while True:
                pid, _ = os.waitpid(-1, os.WNOHANG)
                if pid == 0:
                    break
        except ChildProcessError:
            pass

    def http_status():
        if not health_url:
            return "no probe"
        try:
            with urllib.request.urlopen(health_url, timeout=5) as r:
                return "HTTP {}".format(r.status)
        except (urllib.error.URLError, urllib.error.HTTPError, OSError) as exc:
            return "HTTP DOWN ({})".format(type(exc).__name__)

    def kill_all_vllm():
        for pid in find_vllm_processes(include_zombies=True):
            try:
                os.killpg(os.getpgid(pid), signal.SIGKILL)
                log("  killed leftover pid={}".format(pid))
            except Exception:
                try:
                    os.kill(pid, signal.SIGKILL)
                    log("  killed leftover pid={} (bare kill)".format(pid))
                except Exception as exc:
                    log("  failed to kill pid={}: {}".format(pid, exc))

    def restart():
        state["restart_count"] += 1
        n = state["restart_count"]
        log("RESTART #{}: cleaning up leftovers".format(n))
        kill_all_vllm()
        time.sleep(2)
        reap_zombies()
        log_f = open(log_path, "a", buffering=1)
        log_f.write(
            "\\n=== vllm_watchdog restart #{} at unix={:.0f} ===\\n".format(
                n, time.time()
            )
        )
        log_f.flush()
        proc = subprocess.Popen(
            cmd,
            stdout=log_f,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        log("RESTART #{}: spawned new vllm pid={}".format(n, proc.pid))

    log(
        "active pid={} interval={}s health_url={}".format(
            os.getpid(), check_interval, health_url
        )
    )

    while True:
        try:
            time.sleep(check_interval)
            reap_zombies()
            pids = find_vllm_processes()
            if pids:
                log("healthy: vllm_pids={} ({})".format(pids, http_status()))
            else:
                log("NO LIVE VLLM PROCESS FOUND -- restarting")
                restart()
        except Exception as exc:
            log("loop error {}: {}".format(type(exc).__name__, exc))
            time.sleep(min(check_interval, 10))


if __name__ == "__main__":
    main()
'''

with open(VLLM_WATCHDOG_SCRIPT, 'w') as _f:
    _f.write(_WATCHDOG_SOURCE)

print('Starting vLLM watchdog...')
vllm_watchdog_log = open('vllm_watchdog.log', 'w')
vllm_watchdog_process = subprocess.Popen(
    [sys.executable, VLLM_WATCHDOG_SCRIPT, VLLM_WATCHDOG_CONFIG],
    stdout=vllm_watchdog_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f'vLLM watchdog started (pid {vllm_watchdog_process.pid}).')
print("WALL TIME ELAPSED:", time.time() - GLOBAL_START_TIME)


In [ ]:
# !nvidia-smi

In [ ]:
# !nvidia-smi | grep 'VLLM::EngineCore' | awk '{ print $5 }' | xargs -n1 kill -9

In [ ]:
# !tail /kaggle/working/vllm_watchdog.log
# !nvidia-smi

In [ ]:
import requests

def wait_for_vllm(base_url: str, model_name: str, timeout: float = 120.0, poll_interval: float = 5.0) -> bool:
    """
    Block until vLLM has the model loaded and ready, or timeout expires.
    Uses /v1/models (not /health) — /health returns 200 before weights are loaded.
    Returns True if ready, False if timed out.
    """
    deadline = time.time() + timeout
    attempt = 0

    while time.time() < deadline:
        attempt += 1
        try:
            resp = requests.get(
                f"{base_url}/v1/models",
                timeout=10.0,
            )
            if resp.status_code == 200:
                model_ids = [m["id"] for m in resp.json().get("data", [])]
                if model_name in model_ids:
                    print(f"[vLLM] Ready — model '{model_name}' loaded (attempt {attempt})")
                    return True
                else:
                    print(f"[vLLM] Server up but model not yet listed: {model_ids}")
        except Exception as exc:
            remaining = deadline - time.time()
            print(f"[vLLM] Not ready (attempt {attempt}, {remaining:.0f}s left): {type(exc).__name__}: {exc}")

        time.sleep(poll_interval)

    print(f"[vLLM] Still not ready after {timeout:.0f}s — giving up.")
    return False

In [ ]:
class AIMO3Solver:

    def __init__(self, cfg):

        self.cfg = cfg
        self.client = openai_client
        self.notebook_start_time = time.time()
        self.problems_remaining = 50

    def _compute_mean_entropy(self, logprobs_list: list) -> float:
        """Compute mean entropy from Chat Completions logprobs.

        logprobs_list: flat list of ChatCompletionTokenLogprob objects.
        Each has .top_logprobs = [TopLogprob(token, logprob, bytes), ...].
        """

        if not logprobs_list:
            return float('inf')

        total_entropy = 0.0
        token_count = 0

        for token_logprob in logprobs_list:
            if not token_logprob or not token_logprob.top_logprobs:
                continue

            token_entropy = 0.0

            for tlp in token_logprob.top_logprobs:
                prob = math.exp(tlp.logprob)

                if prob > 0:
                    token_entropy -= prob * math.log2(prob)

            total_entropy += token_entropy
            token_count += 1

        if token_count == 0:
            return float('inf')

        return total_entropy / token_count

    def _process_attempt(
        self,
        problem: str,
        system_prompt: str,
        attempt_index: int,
        stop_event: threading.Event,
        deadline: float
    ) -> dict:

        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1,
                'Answer': None,
                'Python Calls': 0,
                'Python Errors': 0,
                'Response Length': 0,
                'Entropy': float('inf')
            }

        python_calls = 0
        python_errors = 0
        total_completion_tokens = 0
        final_answer = None

        logprobs_buffer = []
        session_id = uuid4().hex
        sandbox = LocalSandbox(host=self.cfg.sandbox_host, port=self.cfg.sandbox_port)

        async def _run_session():
            nonlocal python_calls, python_errors, total_completion_tokens, final_answer

            # Pre-import packages in sandbox
            try:
                await sandbox.execute_code(
                    generated_code=SANDBOX_PREIMPORT,
                    session_id=session_id,
                    timeout=30,
                )
            except Exception:
                pass

            # Build messages
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": problem},
            ]

            extra_body = {}
            if self.cfg.enable_thinking:
                extra_body["chat_template_kwargs"] = {"enable_thinking": True}
                extra_body["skip_special_tokens"] = False

            for turn in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break

                try:
                    stream = self.client.chat.completions.create(
                        model=self.cfg.served_model_name,
                        messages=messages,
                        tools=TOOLS,
                        temperature=self.cfg.temperature,
                        top_p=self.cfg.top_p,
                        logprobs=True,
                        top_logprobs=self.cfg.top_logprobs,
                        stream=True,
                        stream_options={"include_usage": True},
                        **({"extra_body": extra_body} if extra_body else {}),
                    )
                except Exception as exc:
                    print(f'[Attempt {attempt_index + 1}] API error on turn {turn}: {type(exc).__name__}: {exc}')
                    break

                # Accumulate streamed response
                assistant_content = ""
                reasoning_content = ""
                tool_calls_acc = {}
                finish_reason_chunk = None
                turn_completion_tokens = 0

                try:
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break

                        # Usage comes in final chunk (may or may not have choices)
                        if hasattr(chunk, 'usage') and chunk.usage:
                            if getattr(chunk.usage, 'completion_tokens', None):
                                turn_completion_tokens = chunk.usage.completion_tokens

                        choice = chunk.choices[0] if chunk.choices else None
                        if not choice:
                            continue

                        delta = choice.delta

                        if delta.content:
                            assistant_content += delta.content

                        if hasattr(delta, 'reasoning_content') and delta.reasoning_content:
                            reasoning_content += delta.reasoning_content

                        if delta.tool_calls:
                            for tc_delta in delta.tool_calls:
                                idx = tc_delta.index
                                if idx not in tool_calls_acc:
                                    tool_calls_acc[idx] = {
                                        "id": None,
                                        "type": "function",
                                        "function": {"name": None, "arguments": ""},
                                    }
                                entry = tool_calls_acc[idx]
                                if tc_delta.id:
                                    entry["id"] = tc_delta.id
                                if tc_delta.function:
                                    if tc_delta.function.name:
                                        entry["function"]["name"] = tc_delta.function.name
                                    if tc_delta.function.arguments:
                                        entry["function"]["arguments"] += tc_delta.function.arguments

                        if choice.finish_reason:
                            finish_reason_chunk = choice.finish_reason

                        if choice.logprobs and choice.logprobs.content:
                            logprobs_buffer.append(choice.logprobs.content)

                finally:
                    stream.close()

                total_completion_tokens += turn_completion_tokens

                # Finalize tool calls list
                tool_calls = []
                for idx in sorted(tool_calls_acc.keys()):
                    tc = tool_calls_acc[idx]
                    if not tc["id"]:
                        tc["id"] = f"call_{uuid4().hex[:24]}"
                    tool_calls.append(tc)

                # Build assistant message
                assistant_msg = {"role": "assistant", "content": assistant_content or None}
                if tool_calls:
                    assistant_msg["tool_calls"] = tool_calls
                messages.append(assistant_msg)

                # Handle tool calls
                if tool_calls:
                    for tc in tool_calls:
                        python_calls += 1
                        func_args_str = tc["function"]["arguments"]

                        code = ""
                        try:
                            func_args = json.loads(func_args_str)
                            code = func_args.get("code", "")
                        except json.JSONDecodeError:
                            code = func_args_str

                        if code:
                            try:
                                tool_output, _ = await sandbox.execute_code(
                                    generated_code=code,
                                    session_id=session_id,
                                    timeout=self.cfg.python_timeout,
                                    max_output_characters=self.cfg.max_output_characters,
                                )
                                stdout = tool_output.get("stdout", "")
                                stderr = tool_output.get("stderr", "")
                                if stderr:
                                    result = f'{stdout.rstrip()}\n{stderr}' if stdout else stderr
                                else:
                                    result = stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'
                            except Exception as exc:
                                result = f"[ERROR] {type(exc).__name__}: {exc}"
                        else:
                            result = "[ERROR] No code provided."

                        if result.startswith('[ERROR]') or 'Traceback' in result or 'Error' in result:
                            python_errors += 1

                        messages.append({
                            "role": "tool",
                            "name": tc["function"]["name"],
                            "tool_call_id": tc["id"],
                            "content": result,
                        })

                    continue  # Next turn: model responds to tool results

                # No tool calls — check for answer
                text_to_check = _strip_thinking(assistant_content) if assistant_content else ""
                answer = _scan_for_answer(text_to_check)
                if answer is None and assistant_content:
                    answer = _scan_for_answer(assistant_content)
                if answer is not None:
                    final_answer = answer
                break

        try:
            asyncio.run(_run_session())
        except Exception as exc:
            print(f'[Attempt {attempt_index + 1}] Session error: {type(exc).__name__}: {exc}')
            python_errors += 1
        finally:
            try:
                asyncio.run(sandbox.delete_session(session_id))
            except Exception:
                pass
            try:
                asyncio.run(sandbox.close())
            except Exception:
                pass

        # Flatten logprobs for entropy computation
        flat_logprobs = []
        for chunk_logprobs in logprobs_buffer:
            if chunk_logprobs:
                flat_logprobs.extend(chunk_logprobs)

        mean_entropy = self._compute_mean_entropy(flat_logprobs)

        return {
            'Attempt': attempt_index + 1,
            'Response Length': total_completion_tokens,
            'Python Calls': python_calls,
            'Python Errors': python_errors,
            'Entropy': mean_entropy,
            'Answer': final_answer
        }

    def _select_answer(self, detailed_results: list) -> int:

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']

            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)

                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer,
                'votes': answer_votes[answer],
                'score': total_weight
            })

        scored_answers.sort(key=lambda x: x['score'], reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'],
                item['votes'],
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data,
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)

        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer

    def solve_problem(self, problem: str) -> int:
        base_url = f"http://{self.cfg.sandbox_host}:{self.cfg.vllm_port}"
        if not wait_for_vllm(base_url, self.cfg.served_model_name, timeout=120.0):
            print("vLLM unavailable, skipping problem.")
            return 0
    
        print("vLLM server available")
        print(f'\nProblem: {problem}\n')

        user_input = self.cfg.user_template.format(problem=problem)

        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout

        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)

        deadline = time.time() + budget

        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')

        tasks = []

        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))

        detailed_results = []
        valid_answers = []

        stop_event = threading.Event()

        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futures = []

            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt,
                    user_input,
                    system_prompt,
                    attempt_index,
                    stop_event,
                    deadline
                )

                futures.append(future)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)

                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])

                    counts = Counter(valid_answers).most_common(1)

                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()

                        for f in futures:
                            f.cancel()

                        break

                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue

        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)

            self.problems_remaining = max(0, self.problems_remaining - 1)

        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')

            display(results_dataframe)

        if not valid_answers:
            print('\nResult: 0\n')

            return 0

        return self._select_answer(detailed_results)

In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
import random

KAGGLE_HARD_LIMIT = 18000
PANIC_TIME = 30  # return random answer if within 30s of hard kill

def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:

    id_value = id_.item(0)
    question_text = question.item(0)

    # Wall-clock watchdog: if solve_problem hangs for any reason, predict() still returns.
    remaining = GLOBAL_START_TIME + KAGGLE_HARD_LIMIT - PANIC_TIME - time.time()
    if remaining <= 0:
        return pl.DataFrame({'id': id_value, 'answer': random.randint(0, 999)})

    def _solve():
        gc.disable()
        try:
            return solver.solve_problem(question_text)
        finally:
            gc.enable()

    with ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(_solve)
        try:
            final_answer = future.result(timeout=remaining)
        except Exception:
            final_answer = random.randint(0, 999)

    gc.collect()
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [ ]:
import pandas as pd
pd.read_csv('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv').head(3).to_csv('reference_first3.csv', index=False)

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )

In [ ]:
!ls